In [1]:
!pip install kfp-kubernetes==1.4.0

  Preparing metadata (setup.py) ... done
  Created wheel for kfp-kubernetes: filename=kfp_kubernetes-1.4.0-py3-none-any.whl size=20694 sha256=882c40f00bf737f672048a7c719cc8ec0622b84c9f935938c6adfcbeacae6b2f
  Stored in directory: /home/geun-tak.roh-hpe.com/.cache/pip/wheels/16/67/48/190ff78ef59d08f654a51e9509d7c3bd67dc3dd06c23ba4f75
Successfully built kfp-kubernetes


In [1]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [2]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-hp-57f69d47
gl4f-filesystem


## Check Container file-system architecture & workdir

In [3]:
@dsl.component()
def download_dataset(download_url: str, output_file: str, target_path: str):
    import os
    import subprocess
    
    result = subprocess.run(['curl','-L','-o', output_file, download_url],capture_output=True, text=True)
    if result.returncode == 0:
        if os.path.exists(output_file):
            print(f"Successfully downloaded {output_file}")
            print(f"File size: {os.path.getsize(output_file)} bytes")
        else:
            print("Download completed but file not found")
    else:
        print(f"Curl failed with return code {result.returncode}")
        print(f"Error: {result.stderr}")
    
    subprocess.run(['pwd'])
    subprocess.run(['ls','-al'])
    # unzip into the PVC 
    subprocess.run(['unzip',output_file,'-d', target_path])

/opt/conda/lib/python3.11/site-packages/kfp/dsl/component_decorator.py:119: FutureWarning: Python 3.7 has reached end-of-life. The default base_image used by the @dsl.component decorator will switch from 'python:3.7' to 'python:3.8' on April 23, 2024. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.8.
  return component_factory.create_component_from_func(


In [10]:
@dsl.pipeline(
    name="download-dataset"
)
def download_dataset_pipeline(download_url: str, output_file: str, dataset_pvc_name: str, dataset_pvc_size: str,current_sc: str) -> str:
    pvc1 = kubernetes.CreatePVC(
        # can also use pvc_name instead of pvc_name_suffix to use a pre-existing PVC
        pvc_name=dataset_pvc_name,
        access_modes=['ReadWriteMany'],
        size=dataset_pvc_size,
        storage_class_name=current_sc, # gl4fs-system for PCAI
    )
    # write to the PVC
    target_path = '/data'
    task1 = download_dataset(download_url=download_url,output_file=output_file,target_path=target_path)
    kubernetes.mount_pvc(
        task1,
        pvc_name=pvc1.outputs['name'],
        # pvc_name='roboflow-license-plate-datasets',
        mount_path=target_path,
    )
    return pvc1.outputs['name']

In [ ]:
download_url = "" # Enter your url from roboflow
output_file = "/data/roboflow.zip"
dataset_pvc_name = "roboflow-license-plate-datasets-e2e"
dataset_pvc_size = '5Gi'
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()

kfp_client.create_run_from_pipeline_func(
    download_dataset_pipeline,
    arguments={
        'download_url': download_url,
        'output_file': output_file,
        'dataset_pvc_name': dataset_pvc_name,
        'dataset_pvc_size': dataset_pvc_size,
        'current_sc': current_sc
    },
    experiment_name="test-rhgt-exp",
    # enable_caching=False # failed: failed to create PVC and publish execution createpvc: failed to create cache entrty for create pvc: failed to create task: rpc error: code = InvalidArgument desc = Failed to create a new task due to validation error: Invalid input error: Invalid task: must specify FingerPrint
    # enable_caching=True
)

RunPipelineResult(run_id=130b830d-e79a-40f4-be6f-e67ed27c0956)